# Cookbook 1: TabPFN v2: de GBDT para Foundation Models In-Context

**Público-alvo:** graduação em Computação / IA. **Pré-requisito:** scikit-learn básico.

Este notebook tem duas partes:

1. **Teoria** (com vídeo no topo): por que modelos tabulares pré-treinados estão disputando o espaço do gradient boosting.
2. **Prática**: o mesmo protocolo experimental (mesmos dados, mesmos splits, mesma baseline) rodado no **TabPFN v2** e num **RandomForest**, para você comparar número por número.

O protocolo é literalmente o mesmo do `02_mitra_autogluon_cookbook.ipynb` (mesmas partições, mesma baseline, mesmas métricas), então os dois notebooks são comparáveis entre si. Os dois imprimem uma *impressão digital* dos splits e o notebook 02 confere que ela bate.

**Ambiente:** roda no Google Colab (CPU ou GPU) ou em Jupyter local. Sem GPU tudo funciona, só fica mais lento. A primeira execução baixa os pesos do modelo (~100 MB) do Hugging Face.


**Vídeo:** [tutorial prático de TabPFN em Python (Deep Charts)](https://www.youtube.com/watch?v=8RbxKffv_BA). O vídeo usa séries temporais como caso de uso; o mecanismo mostrado é o mesmo que aplicamos aqui em classificação e regressão tabular.


## A transição: por que o boosting deixou de ser a resposta automática

Por cerca de vinte anos, o estado da arte em dados tabulares foi ocupado por árvores com *gradient boosting*: XGBoost, LightGBM, CatBoost. Não foi acaso. Grinsztajn, Oyallon e Varoquaux (2022) mostraram que redes neurais perdiam para essas árvores em dados tabulares por razões estruturais: as colunas são heterogêneas (idade e renda não têm a mesma escala nem a mesma unidade), o sinal útil costuma estar em poucas features e o conjunto de treino é pequeno. Árvores lidam com isso sem esforço; redes neurais precisavam de engenharia de features e tuning que raramente pagavam a conta.

O TabPFN (Hollmann et al., *Nature*, 2025) mudou a pergunta. Em vez de treinar um modelo **para** o seu dataset, treina-se um transformer em milhões de tarefas supervisionadas **sintéticas**. Cada tarefa fictícia nasce de um gerador aleatório (o *prior*), e o modelo aprende a resolver tarefas novas em uma única passada: o conjunto de treino é colocado dentro do contexto (o *support set*), junto com as linhas a prever, e a resposta sai de um *forward pass*. Isso é *in-context learning* (ICL). Note o que desaparece: no momento do `fit` não há descida de gradiente, não há split de validação, não há busca de hiperparâmetro. O ajuste aconteceu todo no pré-treinamento, e o *prior* faz o papel que a validação cruzada fazia.

O TabPFN v2, usado abaixo, é a versão publicada na *Nature*: um transformer com axial attention, pré-treinado para até 10 000 linhas e 500 features, que aceita valores ausentes e features categóricas sem imputação prévia e combina várias passadas (permutações diferentes de features e de codificações posicionais) antes de agregar o resultado. É a mesma família de modelo que você encontra como `REALTABPFN-V2` dentro do AutoGluon.

Três coisas merecem atenção antes de você tirar conclusão do experimento. Primeiro, o ganho é maior justamente onde GBDT sofre: **poucas amostras**. Quando o dataset cresce, o modelo tabular pré-treinado perde vantagem — esse é o limite atacado pelo Mitra, no notebook 02. Segundo, o contexto *é* o modelo: o custo de inferência cresce com o tamanho do *support set* e, fora da distribuição coberta pelo prior, o comportamento degrada sem aviso. Terceiro, o que vem a seguir é **uma única partição de dois datasets pequenos**: serve para você ver o mecanismo com as mãos, não para declarar vencedor. Para isso existem benchmarks com centenas de datasets, como TabArena e TabRepo.


## 1. Ambiente

A célula abaixo instala apenas o que está faltando. No Colab ela baixa o `tabpfn` (que traz o PyTorch); num ambiente local já preparado, ela não faz nada.


In [ ]:
import importlib.util
import subprocess
import sys


def ensure(package: str, import_name: str | None = None) -> None:
    """Install `package` with pip when its module is not importable yet."""
    if importlib.util.find_spec(import_name or package) is None:
        print(f"📦 installing {package} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)


ensure("tabpfn==9.0.0", "tabpfn")  # pinned to the release this cookbook was validated on
print("✅ environment ready")


In [ ]:
import hashlib
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from tabpfn import TabPFNClassifier, TabPFNRegressor
from tabpfn.constants import ModelVersion

try:
    from IPython.display import display
except ImportError:  # plain Python interpreter
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)

# Experimental protocol shared with 02_mitra_autogluon_cookbook.ipynb.
RANDOM_STATE = 42
TEST_SIZE = 0.25
HOUSING_ROWS = 2500
TARGET = "target"

# Pinned on purpose: this cookbook is about TabPFN v2, the Nature release.
# The package default is the newest checkpoint, so we name the version explicitly.
MODEL_VERSION = ModelVersion.V2

# TabPFN v2 refuses more than 1000 training rows when it runs on CPU: it is a performance
# guard, not a model limit (the v2 checkpoint was trained for up to 10 000 rows / 500
# features). The regression split has 1875 training rows, so the guard has to be lifted
# explicitly. On a GPU runtime it never triggers, which makes the flag a no-op there.
TABPFN_KWARGS = {"random_state": RANDOM_STATE, "ignore_pretraining_limits": True}

ARTIFACTS = Path("artifacts_tabpfn")
ARTIFACTS.mkdir(exist_ok=True)
print(f"✅ TabPFN pinned to {MODEL_VERSION.value}")


## 2. Protocolo experimental comum aos dois cookbooks

Tudo o que define o experimento mora aqui e é idêntico no notebook 02: os carregadores, o split e a impressão digital que prova que os dois rodaram sobre as mesmas partições.


In [ ]:
def timeit(fn, *args, **kwargs):
    """Run fn once and return (result, elapsed_seconds)."""
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    return result, time.perf_counter() - start


def fingerprint(*frames) -> str:
    """Order-sensitive hash of the split contents.

    Notebook 02 recomputes the same value and asserts equality: that is how we know the
    two recipes were compared on identical data, and not merely on similar data.
    """
    digest = hashlib.sha256()
    for frame in frames:
        values = np.ascontiguousarray(np.round(frame.to_numpy(dtype=np.float64), 6))
        digest.update(values.tobytes())
    return digest.hexdigest()[:16]


def load_classification_frame() -> pd.DataFrame:
    """Breast cancer: 569 rows, 30 numeric features, binary label in column 'target'."""
    return load_breast_cancer(as_frame=True).frame.rename(columns={"target": TARGET})


def load_regression_frame() -> pd.DataFrame:
    """California housing, subsampled to HOUSING_ROWS rows, label in column 'target'."""
    bunch = fetch_california_housing(as_frame=True)
    return (
        bunch.frame.rename(columns={bunch.target_names[0]: TARGET})
        .sample(n=HOUSING_ROWS, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )


def split_frame(frame: pd.DataFrame, stratify: bool = False):
    """One split, one seed, for both cookbooks."""
    label = frame[TARGET]
    train, test = train_test_split(
        frame,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=label if stratify else None,
    )
    return train.reset_index(drop=True), test.reset_index(drop=True)


def feature_matrix(frame: pd.DataFrame) -> np.ndarray:
    return frame.drop(columns=TARGET).to_numpy(dtype=np.float64)


def classification_metrics(model_name, y_true, proba, y_pred, fit_s, predict_s) -> dict:
    """ROC-AUC and macro-F1 plus the wall-clock cost of fit and predict."""
    return {
        "model": model_name,
        "roc_auc": roc_auc_score(y_true, proba),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "fit_s": round(fit_s, 2),
        "predict_s": round(predict_s, 2),
        "total_s": round(fit_s + predict_s, 2),
    }


def regression_metrics(model_name, y_true, y_pred, fit_s, predict_s) -> dict:
    return {
        "model": model_name,
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "fit_s": round(fit_s, 2),
        "predict_s": round(predict_s, 2),
        "total_s": round(fit_s + predict_s, 2),
    }


RF_KWARGS = {"n_estimators": 300, "n_jobs": -1, "random_state": RANDOM_STATE}

print("✅ helpers ready")


In [ ]:
train_clf, test_clf = split_frame(load_classification_frame(), stratify=True)

print(f"🔎 breast_cancer: train {train_clf.shape} | test {test_clf.shape}")
print(f"   positive rate: train {train_clf[TARGET].mean():.3f} | test {test_clf[TARGET].mean():.3f}")

clf_fingerprint = fingerprint(train_clf, test_clf)
print(f"🧾 split fingerprint (classification): {clf_fingerprint}")


In [ ]:
train_reg, test_reg = split_frame(load_regression_frame())

print(f"🔎 california_housing: train {train_reg.shape} | test {test_reg.shape}")
print(f"   target: mean {train_reg[TARGET].mean():.3f} | std {train_reg[TARGET].std():.3f}")

reg_fingerprint = fingerprint(train_reg, test_reg)
print(f"🧾 split fingerprint (regression): {reg_fingerprint}")

# Contract with notebook 02: it reads this file and asserts the two fingerprints match.
Path("splits_fingerprint.json").write_text(
    json.dumps({"clf": clf_fingerprint, "reg": reg_fingerprint}, indent=2)
)
print("💾 wrote splits_fingerprint.json")


## 3. Classificação: breast cancer

Métricas: **ROC-AUC** (qualidade do ranking de probabilidade, independente do limiar), **F1-macro** (equilíbrio entre precisão e recall, sem favorecer a classe majoritária) e **tempo de parede** de `fit` e `predict`. Nada de `accuracy`: com 62% de positivos, ela esconde erro.

A baseline é um RandomForest: mesmo sendo da família das árvores, ele é o modelo que o aluno de graduação tem à mão em três linhas de scikit-learn, e é o que o TabPFN precisa superar para valer a pena.


In [ ]:
# Baseline: Random Forest, the tree-based reference you already know.
rf_clf = RandomForestClassifier(**RF_KWARGS)
_, rf_clf_fit_s = timeit(rf_clf.fit, feature_matrix(train_clf), train_clf[TARGET].to_numpy())
rf_clf_proba, rf_clf_predict_s = timeit(
    lambda: rf_clf.predict_proba(feature_matrix(test_clf))[:, 1]
)

clf_results = [
    classification_metrics(
        "RandomForest (baseline)",
        test_clf[TARGET].to_numpy(),
        rf_clf_proba,
        rf_clf.predict(feature_matrix(test_clf)),
        rf_clf_fit_s,
        rf_clf_predict_s,
    )
]
display(pd.Series(clf_results[-1]))


In [ ]:
def warm_up_classifier() -> None:
    """First call downloads the checkpoint: keep that out of the measured fit."""
    warm = TabPFNClassifier.create_default_for_version(MODEL_VERSION, **TABPFN_KWARGS)
    warm.fit(feature_matrix(train_clf)[:128], train_clf[TARGET].to_numpy()[:128])


print("🔥 warm-up (downloads the v2 checkpoint, ~100 MB)")
warm_up_classifier()

# TabPFN v2: no gradient step, no validation split. `fit` just preprocesses the support set.
tabpfn_clf = TabPFNClassifier.create_default_for_version(MODEL_VERSION, **TABPFN_KWARGS)
_, tabpfn_clf_fit_s = timeit(
    tabpfn_clf.fit, feature_matrix(train_clf), train_clf[TARGET].to_numpy()
)
tabpfn_clf_proba, tabpfn_clf_predict_s = timeit(
    lambda: tabpfn_clf.predict_proba(feature_matrix(test_clf))[:, 1]
)

clf_results.append(
    classification_metrics(
        "TabPFN v2",
        test_clf[TARGET].to_numpy(),
        tabpfn_clf_proba,
        tabpfn_clf.predict(feature_matrix(test_clf)),
        tabpfn_clf_fit_s,
        tabpfn_clf_predict_s,
    )
)
display(pd.Series(clf_results[-1]))


In [ ]:
clf_table = (
    pd.DataFrame(clf_results).set_index("model").sort_values("roc_auc", ascending=False)
)
clf_table.to_csv(ARTIFACTS / "results_classification.csv")
print("📊 Classification: breast cancer (426 train / 143 test)")
display(clf_table.round(4))


## 4. Regressão: California housing (2500 linhas)

Agora o caso em que o TabPFN v2 ainda joga em casa: 1875 linhas de treino. Métricas de regressão em vez de classificação: **RMSE** (erro em unidades do alvo, penaliza erro grande), **MAE** (erro típico, robusto a outlier) e **R²** (fração da variância explicada, comparável entre modelos).

Amostramos 2500 das 20 640 linhas com `random_state=42` para manter o tempo de CPU sob controle e casar com o experimento do notebook 02.

Um detalhe que você vai encontrar em qualquer ambiente sem GPU: o TabPFN v2 **recusa mais de 1000 linhas de treino quando está na CPU** (`Running on CPU with more than 1000 samples is not allowed by default`). É uma guarda de *performance*, não um limite do modelo: o checkpoint v2 foi treinado para até 10 000 linhas e 500 features. Por isso o construtor abaixo leva `ignore_pretraining_limits=True`; num runtime com GPU a guarda nunca dispara e a flag não muda nada.


In [ ]:
rf_reg = RandomForestRegressor(**RF_KWARGS)
_, rf_reg_fit_s = timeit(rf_reg.fit, feature_matrix(train_reg), train_reg[TARGET].to_numpy())
rf_reg_pred, rf_reg_predict_s = timeit(rf_reg.predict, feature_matrix(test_reg))

reg_results = [
    regression_metrics(
        "RandomForest (baseline)",
        test_reg[TARGET].to_numpy(),
        rf_reg_pred,
        rf_reg_fit_s,
        rf_reg_predict_s,
    )
]
display(pd.Series(reg_results[-1]))


In [ ]:
def warm_up_regressor() -> None:
    warm = TabPFNRegressor.create_default_for_version(MODEL_VERSION, **TABPFN_KWARGS)
    warm.fit(feature_matrix(train_reg)[:128], train_reg[TARGET].to_numpy()[:128])


print("🔥 warm-up (loads the v2 regression checkpoint)")
warm_up_regressor()

tabpfn_reg = TabPFNRegressor.create_default_for_version(MODEL_VERSION, **TABPFN_KWARGS)
_, tabpfn_reg_fit_s = timeit(
    tabpfn_reg.fit, feature_matrix(train_reg), train_reg[TARGET].to_numpy()
)
tabpfn_reg_pred, tabpfn_reg_predict_s = timeit(
    tabpfn_reg.predict, feature_matrix(test_reg)
)

reg_results.append(
    regression_metrics(
        "TabPFN v2",
        test_reg[TARGET].to_numpy(),
        tabpfn_reg_pred,
        tabpfn_reg_fit_s,
        tabpfn_reg_predict_s,
    )
)
display(pd.Series(reg_results[-1]))


In [ ]:
reg_table = (
    pd.DataFrame(reg_results).set_index("model").sort_values("rmse", ascending=True)
)
reg_table.to_csv(ARTIFACTS / "results_regression.csv")
print("📊 Regression: california_housing, 2500 rows sampled (1875 train / 625 test)")
display(reg_table.round(4))


## 5. Como ler o que você acabou de ver

Compare **não só a métrica, mas a coluna `fit_s`**. O RandomForest *aprendeu* algo a partir dos dados de treino; o TabPFN apenas empacotou o treino como contexto. Onde está o custo, então? No `predict_s`: quanto maior o *support set*, mais caro cada forward pass. Esse é o trade-off que você vai encontrar em produção, e é o motivo de modelos ICL serem servidos com cache de contexto.

Em dados deste tamanho, o esperado é que o modelo pré-treinado empate ou ganhe com pouquíssimo esforço de tuning. Se o seu caso tem 100 000 linhas, essa vantagem desaparece: aí a resposta continua sendo boosting, ou um *foundation model* desenhado para escala, como o Mitra do notebook 02.

Um aviso metodológico: uma partição só não sustenta conclusão. Se quiser afirmar algo, repita com `random_state` variados ou use `TabPFNClassifier(n_estimators=...)` com validação por folds externos e reporte média e desvio — e compare contra GBDT de verdade (LightGBM/CatBoost), não contra um RandomForest.

### Referências

**Artigo principal (TabPFN)**
Hollmann, N., Müller, S., Purucker, L., Krishnakumar, A., Körfer, M., Hoo, S. B., Schirrmeister, R. T., & Hutter, F. (2025). *Accurate predictions on small data with a tabular foundation model.* **Nature** 637, 319–326. DOI: 10.1038/s41586-024-08328-6. Preprint: arXiv:2402.12884.

**Por que GBDT dominava dados tabulares**
Grinsztajn, L., Oyallon, E., & Varoquaux, G. (2022). *Why do tree-based models still outperform deep learning on typical tabular data?* NeurIPS 2022, Datasets and Benchmarks Track. arXiv:2207.08815.

**Modelo comparado no notebook 02 (Mitra)**
Zhang, X., Maddix, D. C., Yin, J., Erickson, N., Ansari, A. F., Han, B., Zhang, S., Akoglu, L., Faloutsos, C., Mahoney, M. W., Hu, C., Rangwala, H., Karypis, G., & Wang, B. (2025). *Mitra: Mixed Synthetic Priors for Enhancing Tabular Foundation Models.* NeurIPS 2025. arXiv:2510.21204.

**Documentação e código**
TabPFN: https://github.com/PriorLabs/TabPFN · documentação: https://docs.priorlabs.ai · pesos no Hugging Face: https://huggingface.co/Prior-Labs

```bibtex
@article{hollmann2025tabpfn,
  title   = {Accurate predictions on small data with a tabular foundation model},
  author  = {Hollmann, Noah and M{\"u}ller, Samuel and Purucker, Lennart and
             Krishnakumar, Arjun and K{\"o}rfer, Max and Hoo, Shi Bin and
             Schirrmeister, Robin Tibor and Hutter, Frank},
  journal = {Nature},
  volume  = {637},
  pages   = {319--326},
  year    = {2025},
  doi     = {10.1038/s41586-024-08328-6}
}

@inproceedings{grinsztajn2022trees,
  title     = {Why do tree-based models still outperform deep learning on typical tabular data?},
  author    = {Grinsztajn, L{\'e}o and Oyallon, Edouard and Varoquaux, Ga{\"e}l},
  booktitle = {Advances in Neural Information Processing Systems (NeurIPS), Datasets and Benchmarks Track},
  year      = {2022},
  eprint    = {2207.08815},
  archivePrefix = {arXiv}
}

@inproceedings{zhang2025mitra,
  title     = {Mitra: Mixed Synthetic Priors for Enhancing Tabular Foundation Models},
  author    = {Zhang, Xiyuan and Maddix, Danielle C. and Yin, Junming and Erickson, Nick and
               Ansari, Abdul Fatir and Han, Boran and Zhang, Shuai and Akoglu, Leman and
               Faloutsos, Christos and Mahoney, Michael W. and Hu, Cuixiong and
               Rangwala, Huzefa and Karypis, George and Wang, Bernie},
  booktitle = {Advances in Neural Information Processing Systems (NeurIPS)},
  year      = {2025},
  eprint    = {2510.21204},
  archivePrefix = {arXiv}
}
```

**Próximo passo:** `02_mitra_autogluon_cookbook.ipynb`, mesmo protocolo, com o modelo `MITRA` do AutoGluon.
